# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iHafizah/my-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking (built on a score).**
The model gives every page a priority score, and then I sort the pages from highest to lowest. The page at the top is the one the team should check first. I chose ranking because the team can only review a few pages, so the order matters more than the exact number.

In [1]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Number of pages:", len(df))

Number of pages: 30000


## 2. Target or proxy

**Target (proxy): whether a page's traffic is falling.**
I want to find pages in trouble, but the data has no column that says that. Falling traffic is the closest sign, so I use it as a stand-in. The label comes from an observed outcome, because the trend is already recorded in the data. It isn't perfect: it is measured over the current window and not a future outcome, and traffic can also fall because of season or low demand.

In [2]:
df["is_declining_label"] = df["trend_direction"] == "down"
print(df["is_declining_label"].value_counts())

is_declining_label
True     16262
False    13738
Name: count, dtype: int64


## 3. Success metric

**Success metric: Precision@50.**
Of the 50 pages ranked highest, how many really had falling traffic? A higher share means the top of the list is more useful. I chose it because the team can only review a few pages, so the top of the list matters most, and plain accuracy over all 30,000 pages would hide that. I would compare it with the overall share of falling pages, because a list is only useful if it beats picking pages at random.

No model is built yet. The code below ranks pages with a simple "most seen first" rule, only to show how the metric is calculated. A real model's ranking would replace it later.

In [3]:
top50 = df.sort_values("impressions_90d", ascending=False).head(50)
print("Precision@50 of a simple 'most seen first' list:", top50["is_declining_label"].mean())
print("Share of falling pages overall (random-pick level):", df["is_declining_label"].mean())


Precision@50 of a simple 'most seen first' list: 0.42
Share of falling pages overall (random-pick level): 0.5420666666666667


## 4. The unit of analysis, as a real dataframe

**One row = one page.**
Each row in the table is a single web page (a content item), with signals about it such as impressions, sessions, age and traffic trend. The last column, is_declining_label, is the target: True if the page's traffic is falling. The table below shows the first rows.

In [4]:
cols = ["content_id", "client_id", "impressions_90d", "sessions_90d",
        "content_age_days", "trend_direction", "is_declining_label"]
df[cols].head()


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,187,down,True
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down,True
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down,True
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable,False
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down,True


## 5. Why ML beats a fixed rule here

**Why ML could beat a simple if/else rule here.**
An if/else rule is a line drawn by hand, like "if impressions are above the median, flag the page". It has three problems. It looks at only one or two signals. Its cut-offs are guesses (in my first notebook, a 180-day cut-off matched only 15 of 30,000 pages). And it only says yes or no, so the team gets a long unordered list: this rule flags about half of the 30,000 pages. It also does not work well: the simple "most seen first" list scored 0.42, below the 0.54 overall share, so it did worse than picking pages at random.

A model could combine many signals (impressions, sessions, age, clicks), learn the cut-offs from the data, and give every page a score, so the pages come out ranked and the team reads the top.

This is the reason to try ML, not proof that it works. I have not built a model, so I cannot claim it wins. I would only believe it if its Precision@50 beat simple rules and the 0.54 random-pick level.


In [5]:
rule = df["impressions_90d"] > df["impressions_90d"].median()
print("Pages the fixed rule flags:", rule.sum())
print("Share of flagged pages with falling traffic:", df.loc[rule, "is_declining_label"].mean())
print("Share of falling pages overall:", df["is_declining_label"].mean())


Pages the fixed rule flags: 14998
Share of flagged pages with falling traffic: 0.5938125083344445
Share of falling pages overall: 0.5420666666666667


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.